DMP PDF
  ↓
1. pdfplumber extraction
  - extract line-level text
  - keep page number, font size, bold, position

  ↓
2. rule-based structure detection
  - Element 1, Element 2 → section
  - A., B., C. → question
  - remaining lines → answer/content

  ↓
3. build narrative JSON
  - load your full RDA + DMPTool extension skeleton
  - keep metadata fields as null
  - fill only narrative.template.section

  ↓
4. save final JSON
  - output looks like your skeleton
  - narrative part contains extracted DMP sections/questions/answers

PDF line label              JSON location
--------------------------------------------------
section                     narrative.template.section[].title

subsection                  narrative.template.section[].question[].text

content after subsection    question[].answer.json.answer[].text

In [1]:
from pathlib import Path
import json
import pandas as pd

from dmpbridge.pdf.pdfplumber_extractor import save_pdfplumber_outputs
from dmpbridge.processing.structure_detector import detect_structure
from dmpbridge.processing.structure_json_builder import save_narrative_json

In [2]:
project_root = Path.cwd().parent

pdf_path = project_root / "data" / "raw_pdfs" /"sample1.pdf"
skeleton_path = project_root / "schemas" / "rda_dmp_dmptool_extension_skeleton.json"

pdfplumber_json_path = project_root / "data" / "pdfplumber_blocks" / f"{pdf_path.stem}.json"
csv_output_path = project_root / "outputs" / "debug" / f"{pdf_path.stem}_structured_lines.csv"
final_json_path = project_root / "data" / "structure_json" / f"{pdf_path.stem}_narrative.json"

print("PDF exists:", pdf_path.exists())
print("Skeleton exists:", skeleton_path.exists())

PDF exists: True
Skeleton exists: True


In [3]:
blocks = save_pdfplumber_outputs(pdf_path)

print("Extracted lines:", len(blocks))
print("Saved pdfplumber JSON:", pdfplumber_json_path.exists())

[2026-05-05 07:42:48] Extracting line-level text with pdfplumber: sample1.pdf
[2026-05-05 07:42:48] Saved line-level JSON: C:\Users\Nahid\dmpbridge\data\pdfplumber_blocks\sample1.json
[2026-05-05 07:42:48] Saved extracted text: C:\Users\Nahid\dmpbridge\data\extracted_text\sample1.txt
Extracted lines: 79
Saved pdfplumber JSON: True


In [4]:
structured_blocks = detect_structure(blocks)

df = pd.DataFrame(structured_blocks)

df[["page", "line_order", "text", "avg_font_size", "is_bold", "label"]].head(50)

,page,line_order,text,avg_font_size,is_bold,label
0,1,1,DATA MANAGEMENT AND SHARING PLAN,11.04,True,section
1,1,2,Element 1: Data Type:,11.04,True,section
2,1,3,A. Types and amount of scientific data expecte...,11.04,True,subsection
3,1,4,This secondary data analysis project will anal...,11.04,False,content
4,1,5,and the publicly available NHANES cohorts (wri...,11.04,False,content
5,1,6,"The studies include (i) the RISE Study, (ii) t...",11.04,False,content
6,1,7,"(v) the PHASE Study, (vi) the AusDiab Study, (...",11.04,False,content
7,1,8,study.,11.04,False,content
8,1,9,B. Scientific data that will be preserved and ...,11.04,True,subsection
9,1,10,"As this is a secondary data analysis project, ...",11.04,False,content


In [5]:
csv_output_path.parent.mkdir(parents=True, exist_ok=True)

df[["page", "line_order", "text", "avg_font_size", "is_bold", "label"]].to_csv(
    csv_output_path,
    index=False,
    encoding="utf-8"
)

print("Saved CSV:", csv_output_path)

Saved CSV: c:\Users\Nahid\dmpbridge\outputs\debug\sample1_structured_lines.csv


In [6]:
final_json = save_narrative_json(
    structured_blocks=structured_blocks,
    output_path=final_json_path,
    skeleton_path=skeleton_path
)

print("Saved final JSON:", final_json_path)
print("Number of sections:", len(final_json["narrative"]["template"]["section"]))

[2026-05-05 07:42:59] Saved narrative JSON: c:\Users\Nahid\dmpbridge\data\structure_json\sample1_narrative.json
Saved final JSON: c:\Users\Nahid\dmpbridge\data\structure_json\sample1_narrative.json
Number of sections: 7


In [7]:
for section in final_json["narrative"]["template"]["section"]:
    print(section["order"], section["title"], "| questions:", len(section["question"]))

1 DATA MANAGEMENT AND SHARING PLAN | questions: 0
2 Element 1: Data Type: | questions: 3
3 Element 2: Related Tools, Software and/or Code: | questions: 0
4 Element 3: Standards: | questions: 0
5 Element 4: Data Preservation, Access, and Associated Timelines: | questions: 3
6 Element 5: Access, Distribution, or Reuse Considerations: | questions: 2
7 Element 6: Oversight of Data Management and Sharing: | questions: 0


In [8]:
print(final_json_path.exists())
print(final_json_path)

True
c:\Users\Nahid\dmpbridge\data\structure_json\sample1_narrative.json
